# this is training the CNNPZ model on the noisy mock data with randomly dropping bands

In [ ]:
import sys

from packaging import version
import sklearn
from sklearn.model_selection import KFold, train_test_split

assert version.parse(sklearn.__version__) >= version.parse("1.0.1")

import tensorflow as tf

assert version.parse(tf.__version__) >= version.parse("2.8.0")

#import tensorflow_probability as tfp

import matplotlib.pyplot as plt

plt.rc('font', size=14)
plt.rc('axes', labelsize=14, titlesize=14)
plt.rc('legend', fontsize=14)
plt.rc('xtick', labelsize=10)
plt.rc('ytick', labelsize=10)

import pandas as pd
import h5py
import numpy as np

import json
import os

from matplotlib import colors

In [ ]:
import cnnpz

In [ ]:
# Parametric paths (edit via environment variables, or defaults below)
USER = os.environ.get("USER", "jaimerz")
PSCRATCH = os.environ.get("PSCRATCH", f"/pscratch/sd/{USER[0]}/{USER}")
PROJECT_HOME = os.environ.get("CNNPZ_PROJECT_HOME", f"/global/homes/{USER[0]}/{USER}/UCL")

# Local data folder (train/test parquet files + filter curves live here together)
LOCAL_DATA_ROOT = os.environ.get("CNNPZ_LOCAL_DATA_ROOT", os.path.join(os.getcwd(), "data"))

CARDINAL_DATA_ROOT = LOCAL_DATA_ROOT + "/"
MODEL_ROOT = os.path.join(PSCRATCH, "cnnpz", "noisy_Cardinal", "models")
PRETRAINING_DATA_ROOT = os.path.join(PSCRATCH, "pop-cosmos-data")
FILTER_ROOT = LOCAL_DATA_ROOT + "/"


## Load training and test data, filter curves

In [ ]:
saveroot = CARDINAL_DATA_ROOT
fname = saveroot + "train_100k_noisy_y1_i23.parquet"
training_y1 = pd.read_parquet(fname)
training_y1 = training_y1.rename(columns={'Roman_obs_Y106': 'mag_Y_roman',
                                              'Roman_obs_J129': 'mag_J_roman', 
                                              'Roman_obs_H158': 'mag_H_roman'})

fname = saveroot + "train_100k_noisy_y10_i25.4.parquet"
training_y10 = pd.read_parquet(fname)
training_y10 = training_y10.rename(columns={'Roman_obs_Y106': 'mag_Y_roman',
                                              'Roman_obs_J129': 'mag_J_roman', 
                                              'Roman_obs_H158': 'mag_H_roman'})

saveroot = CARDINAL_DATA_ROOT
fname = saveroot + "test_100k_noisy_y1_i23.parquet"
test_y1 = pd.read_parquet(fname)
test_y1 = test_y1.rename(columns={'Roman_obs_Y106': 'mag_Y_roman',
                                              'Roman_obs_J129': 'mag_J_roman', 
                                              'Roman_obs_H158': 'mag_H_roman'})

fname = saveroot + "test_100k_noisy_y10_i25.4.parquet"
test_y10 = pd.read_parquet(fname)
test_y10 = test_y10.rename(columns={'Roman_obs_Y106': 'mag_Y_roman',
                                              'Roman_obs_J129': 'mag_J_roman', 
                                              'Roman_obs_H158': 'mag_H_roman'})

### Transform data, make X_train, Y_train, and test dataset, variants of this dataset

In [ ]:
# get the LSST and roman filter curves:
filter_root = FILTER_ROOT

wave = {
    "Y":106,
    "J":129,
    "H":158,
}
lsst_filter_curves = {}
roman_filter_curves = {}
for b in "ugrizy":
  lsst_filter_curves[b] = np.loadtxt(filter_root + f'DC2LSST_{b}.res')

for b in "YJH":
  roman_filter_curves[b] = np.loadtxt(filter_root + f'roman_{b}{wave[b]}.res')

lambda_min = lsst_filter_curves['u'][:,0].min()
lambda_max = roman_filter_curves['H'][:,0].max()
print(lambda_min, lambda_max)

## Continuous filter-curve representation

Each band's real transmission curve is interpolated onto a shared wavelength grid and area-normalized, then a galaxy's photometry is turned into a continuous curve over wavelength and binned to 32 points to match the CNN's input length.

$$
T_b(\lambda) = \text{interpolated raw transmission curve of band } b \text{ on the common grid}
$$

$$
\text{filters\_array}_b(\lambda) = \frac{T_b(\lambda)}{\int T_b(\lambda)\, d\lambda}
$$

Within wavelength bin $i$, each band's area-normalized curve is integrated and the bin's weights are rescaled to sum to 1 across bands, giving a proper weighted-average magnitude per bin:

$$
\text{filters\_binned}_{b,i} = \frac{\int_{\text{bin } i} \text{filters\_array}_b(\lambda)\, d\lambda}{\sum_{b'} \int_{\text{bin } i} \text{filters\_array}_{b'}(\lambda)\, d\lambda}
$$

The coverage channel just checks, per galaxy, whether any *observed* band contributes to a given bin (filters_binned > 0 there) -- binary, no separate ownership operator needed:

$$
\text{coverage}_i = \mathbb{1}\left[\sum_b o_b \cdot \mathbb{1}[\text{filters\_binned}_{b,i} > 0] > 0\right], \qquad o_b \in \{0,1\}
$$


In [ ]:
roman_bands = "YJH"
lsst_bands = "ugrizyYJH"
mag_columns = [f"mag_{b}_roman" if b in roman_bands else f"mag_{b}_lsst" for b in lsst_bands]
mags = np.array(test_y1[mag_columns])
mags.shape

In [ ]:
# build the filter bank (avoids double-counting overlapping filters) and bin it
# down to the CNN's 32 input bins -- 9 bands now, bringing back Roman Y106
lambda_common = np.linspace(lambda_min, lambda_max, 1000)
bands = "ugrizyYJH"
filter_curves = {**lsst_filter_curves, **roman_filter_curves}
filters_array, ownership = cnnpz.interpolate_filter_curves(filter_curves, lambda_common)

n_lambda_bins = 32
lambda_bin_centers, _ = cnnpz.make_lambda_bins(lambda_common, n_lambda_bins)
filters_binned = cnnpz.bin_filters(filters_array, n_lambda_bins, lambda_common)
print("filters_array:", filters_array.shape, "filters_binned:", filters_binned.shape)

# cnnpz's data-prep functions take plain (n_filters, n_sources) magnitude arrays instead
# of a dataframe -- this bridges our per-project column naming to that.
roman_bands = "YJH"
def mags_from_df(df, bands):
    cols = [f"mag_{b}_roman" if b in roman_bands else f"mag_{b}_lsst" for b in bands]
    return df[cols].to_numpy().T  # (n_filters, n_sources)

i_idx = bands.index("i")
nir_idx = np.array([bands.index(b) for b in "JH"])

In [ ]:
plt.plot(filters_array.T)
plt.show()

In [ ]:
# ensure a clean positional index before building datasets -- train_ensembles indexes
# Y positionally via KFold, which silently breaks on a non-default index
training_y1 = training_y1.reset_index(drop=True)
test_y1 = test_y1.reset_index(drop=True)

mags_train_y1 = mags_from_df(training_y1, bands)
mags_test_y1 = mags_from_df(test_y1, bands)

X_y1, Y_y1 = cnnpz.transform_data_to_XY(
    mags_train_y1,
    training_y1["redshift"].to_numpy(),
    filters_array,
    ownership,
    n_lambda_bins,
    lambda_common,
    mag_i=mags_train_y1[i_idx],
    apply_stretch=False)
X_test_y1, Y_test_y1 = cnnpz.transform_data_to_XY(
    mags_test_y1,
    test_y1["redshift"].to_numpy(),
    filters_array,
    ownership,
    n_lambda_bins,
    lambda_common,
    mag_i=mags_test_y1[i_idx],
    apply_stretch=False)
print(X_y1.shape, Y_y1.shape)

## Visualize and train the model (Y1)

In [ ]:
cnnpz.visualize_the_data(X_y1, Y_y1, lambda_bin_centers, title="Y1 example")

In [ ]:
# train on Y1 complete (or load if already trained)
from cnnpz import build_model
save_dir = "./models/y1_curve_9band_ensemble_CNN_6layers"
if os.path.exists(os.path.join(save_dir, "norm_params.json")):
    trained_models = cnnpz.load_ensemble(save_dir=save_dir)
    histories = None
else:
    trained_models, histories = cnnpz.train_ensembles(build_model, X_y1, Y_y1)
    cnnpz.save_ensemble(trained_models, save_dir=save_dir)

In [ ]:
if histories is not None:
    cnnpz.plot_ensemble_losses(histories, ylim=(0, 0.001))

In [ ]:
# Final prediction on test set
y_pred_ensemble, y_pred_STD = cnnpz.ensemble_predict(trained_models, X_test_y1)

In [ ]:
redshift_bins = np.linspace(0,2.5,11)
imag_bins = np.linspace(18, 25.5,11)
stats2, redshift_stats2, imag_stats2 = cnnpz.get_all_stats(
    Y_test_y1, y_pred_ensemble.flatten(), test_y1['mag_i_lsst'], save=False,
    redshift_bins=redshift_bins, imag_bins=imag_bins)

cnnpz.plot_stats(stats2, redshift_stats2, imag_stats2, Y_test_y1, y_pred_ensemble.flatten(),
           redshift_bins, imag_bins, test_y1['mag_i_lsst'], save_path=save_dir + '/photoz_stats.png')